**GeoBarangay AI**

In [1]:
import os
import sys
import cx_Oracle
from sqlalchemy import create_engine
import pandas as pd

try:
    # For Windows User
    if sys.platform.startswith("win32"):
        lib_dir = r"C:\Oracle\instantclient_19_24"
        cx_Oracle.init_oracle_client(lib_dir=lib_dir)
except cx_Oracle.ProgrammingError as err:
    if "already been initialized" in str(err):
        print("Oracle Client library has already been initialized. Continuing...")
    else:
        print("Whoops!")
        print(err)
        sys.exit(1)
except Exception as err:
    print("Whoops!")
    print(err)
    sys.exit(1)

# Connect to PostgreSQL
pg_user = 'pguser'
pg_password = 'user'
pg_host = '35.247.173.6'
pg_port = '5432'
pg_database = 'postgres'
pg_connection_string = f'postgresql://{pg_user}:{pg_password}@{pg_host}:{pg_port}/{pg_database}'
pg_engine = create_engine(pg_connection_string)

In [3]:
customer_no_brgy = """ select "id",cus_latitude,cus_longitude,'Barangay' from dlpc_gis.customer_info where classification is null
"""

new_df = pd.read_sql_query(customer_no_brgy, pg_engine)

In [4]:
import pandas as pd
import joblib
import numpy as np

# Path to the .pkl files on the new PC
model_path = r"C:\Users\OMontero\Downloads\GeBarangay AI\random_forest_model.pkl"  # Adjust the path as necessary
scaler_path = r"C:\Users\OMontero\Downloads\GeBarangay AI\scaler.pkl" # Adjust the path as necessary

# Load the saved model and scaler
model = joblib.load(model_path)
scaler = joblib.load(scaler_path)

# Load new data
new_csv_file_path = r"C:\Users\OMontero\Downloads\GeBarangay AI\customer_no_brgy.csv" # Adjust the path as necessary
new_df = pd.read_csv(new_csv_file_path)

# Preprocess the new data
new_df.replace([np.inf, -np.inf], np.nan, inplace=True)
new_df.dropna(subset=['cus_latitude', 'cus_longitude'], inplace=True)  # Ensure necessary columns are not NaN

# Select features
X_new = new_df[['cus_latitude', 'cus_longitude']]

# Scale the features using the loaded scaler
X_new_scaled = scaler.transform(X_new)

# Use the model to make predictions
predictions = model.predict(X_new_scaled)

# Add predictions to the DataFrame
new_df['predicted_barangay'] = predictions

# Load the mapping CSV
mapping_csv_path = r"C:\Users\OMontero\Downloads\GeBarangay AI\ai_brgy_keylist.csv" # Path to your mapping file
mapping_df = pd.read_csv(mapping_csv_path)

# Ensure the merge keys are of the same type
new_df['predicted_barangay'] = new_df['predicted_barangay'].astype(str)
mapping_df['ID'] = mapping_df['ID'].astype(str)

# Merge the new DataFrame with the mapping DataFrame to get the barangay names
result_df = new_df.merge(mapping_df, left_on='predicted_barangay', right_on='ID', how='left')
result_df = result_df.drop(columns=['barangay', 'ID', 'predicted_barangay'])

# Save the DataFrame with the predicted barangay names
#result_df.to_csv(r"C:\Users\OMontero\Downloads\new_data_with_predictions_final.csv", index=False)

print("Predictions with barangay names have been saved to 'new_data_with_predictions_final.csv'.")


Predictions with barangay names have been saved to 'new_data_with_predictions_final.csv'.


In [6]:
result_df = result_df.drop(columns=['cus_latitude','cus_longitude','delivery_address'])


In [8]:
try:
    result_df.to_sql('customer_info_classification', pg_engine, schema='dlpc_gis', if_exists='append', index=False)
    print("Data inserted successfully!")
except Exception as e:
    print(f"An error occurred: {e}")

from sqlalchemy import create_engine, text

Data inserted successfully!


In [9]:
from sqlalchemy import create_engine, text


update_classification_brgy = """UPDATE dlpc_gis.customer_info x
SET corrected_brgy = y."Barangay",
    classification = y.classification
FROM dlpc_gis.customer_info_classification y
WHERE x.cus_latitude IS NULL 
  AND x."id"  = y."id" 
"""


# Execute the SQL UPDATE statements
with pg_engine.connect() as connection:
    with connection.begin() as transaction:
        try:
            # Insert to update from customer_update to customer_info
            connection.execute(text(update_classification_brgy))
         

            # Commit the transaction
            transaction.commit()
        except Exception as e:
            print(f"An error occurred: {e}")
            transaction.rollback()


print('CUSTOMER Script Successfully Executed')

CUSTOMER Script Successfully Executed


**Average Latlong for each barangay to check**

In [16]:
import pandas as pd
import joblib
import numpy as np

# Path to the .pkl files on the new PC
model_path = r"C:\Users\OMontero\Downloads\GeBarangay AI\random_forest_model.pkl"  # Adjust the path as necessary
scaler_path = r"C:\Users\OMontero\Downloads\GeBarangay AI\scaler.pkl"  # Adjust the path as necessary

# Load the saved model and scaler
model = joblib.load(model_path)
scaler = joblib.load(scaler_path)

# Load new data
new_csv_file_path = r"C:\Users\OMontero\Downloads\GeBarangay AI\customer_no_brgy.csv"  # Adjust the path as necessary
new_df = pd.read_csv(new_csv_file_path)

# Preprocess the new data
new_df.replace([np.inf, -np.inf], np.nan, inplace=True)
new_df.dropna(subset=['cus_latitude', 'cus_longitude'], inplace=True)  # Ensure necessary columns are not NaN

# Select features
X_new = new_df[['cus_latitude', 'cus_longitude']]

# Scale the features using the loaded scaler
X_new_scaled = scaler.transform(X_new)

# Use the model to make predictions
predictions = model.predict(X_new_scaled)

# Add predictions to the DataFrame
new_df['predicted_barangay'] = predictions

# Load the mapping CSV
mapping_csv_path = r"C:\Users\OMontero\Downloads\GeBarangay AI\ai_brgy_keylist.csv"  # Path to your mapping file
mapping_df = pd.read_csv(mapping_csv_path)

# Ensure the merge keys are of the same type
new_df['predicted_barangay'] = new_df['predicted_barangay'].astype(str)
mapping_df['ID'] = mapping_df['ID'].astype(str)

# Merge the new DataFrame with the mapping DataFrame to get the barangay names
result_df = new_df.merge(mapping_df, left_on='predicted_barangay', right_on='ID', how='left')

# Drop unnecessary columns
result_df = result_df.drop(columns=['barangay', 'predicted_barangay'])

# Save the DataFrame with the predicted barangay names
result_csv_path = r"C:\Users\OMontero\Downloads\new_data_with_predictions_final.csv"
result_df.to_csv(result_csv_path, index=False)

print(f"Predictions with barangay names have been saved to '{result_csv_path}'.")

# Group by ID and calculate the average latitude and longitude
average_latlong_per_id_df = result_df.groupby('ID').agg(
    avg_latitude=('cus_latitude', 'mean'),
    avg_longitude=('cus_longitude', 'mean')
).reset_index()

# Save the average latitude and longitude for each ID to a CSV
average_latlong_id_csv_path = r"C:\Users\OMontero\Downloads\GeBarangay AI\average_latlong_per_id.csv"
average_latlong_per_id_df.to_csv(average_latlong_id_csv_path, index=False)

print(f"Average latitude and longitude for each ID have been saved to '{average_latlong_id_csv_path}'.")

Predictions with barangay names have been saved to 'C:\Users\OMontero\Downloads\new_data_with_predictions_final.csv'.
Average latitude and longitude for each ID have been saved to 'C:\Users\OMontero\Downloads\GeBarangay AI\average_latlong_per_id.csv'.
